# Full pipeline on your own data (no scraping step)

Runs stages 2-11 of this project's pipeline -- chunking through evaluation
metrics -- against an already-prepared CSV of your own, skipping the
sourcing/scraping stage entirely (that only applies to `fln`/`euaa`/`asylex`,
via `full_pipeline_generic.ipynb`).

Fill in `INPUT_CSV`, `TEXT_COLUMN`, and `STEM` in the first cell, then run
the notebook top to bottom.

**Every stage is idempotent**: it loads the existing output file if one is
already on disk, and only computes it otherwise -- safe to re-run.

**Kernel:** must run under a Python distribution where PyTerrier's embedded
JVM actually starts (Anaconda's, on this machine) -- retrieval and
evaluation need it; the earlier stages don't.

## 0. Setup

In [ ]:
import os, sys
import pandas as pd

PROJECT_DIR = os.path.abspath(".")
SRC_DIR = os.path.join(PROJECT_DIR, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

pd.set_option("display.max_colwidth", 80)

INPUT_CSV = "data/my_data.csv"      # <- your own already-scraped/sampled data
TEXT_COLUMN = "my_text_col"         # <- whichever column holds the document text
STEM = "my_data"                    # <- prefix every derived filename will use

def step_done(path):
    return os.path.exists(path) and os.path.getsize(path) > 0

## 1. Chunk the documents

`chunk_documents.py`'s `DocumentChunker`: splits each document at natural
paragraph/sentence boundaries into ~400-char chunks.

In [ ]:
from chunk_documents import ChunkingConfig, DocumentChunker

CHUNKS_CSV = f"data/{STEM}_chunks.csv"
if not step_done(CHUNKS_CSV):
    config = ChunkingConfig(input_csv=INPUT_CSV, text_column=TEXT_COLUMN, output_csv=CHUNKS_CSV)
    DocumentChunker(config).run()

df_chunks = pd.read_csv(CHUNKS_CSV)
print(f"{df_chunks['doc_id'].nunique()} documents -> {len(df_chunks)} chunks")
df_chunks.head(3)

## 2. Generate one query per chunk (few-shot, local LLM)

`generate_queries.py`'s `QueryGenerator`, via a local Ollama model
(`gemma3:4b` default). Slow on a full dataset -- skipped here if already
done.

In [ ]:
from generate_queries import QueryGenConfig, QueryGenerator

QUERIES_CSV = f"data/{STEM}_queries.csv"
if not step_done(QUERIES_CSV):
    print("Generating one query per chunk via Ollama (this can take a while)...")
    config = QueryGenConfig(input_csv=CHUNKS_CSV, output_csv=QUERIES_CSV)
    QueryGenerator(config).run()

df_queries = pd.read_csv(QUERIES_CSV)
print(f"{len(df_queries)} generated queries")
df_queries[["chunk_id", "generated_query"]].head(3)

## 3. Cluster queries into topics (BERTopic) and name each with an LLM

`extract_topics.py`'s `QueryTopicExtractor`: sentence-transformers embedding
-> UMAP -> HDBSCAN -> c-TF-IDF keywords -> Ollama-generated topic name.

In [ ]:
from extract_topics import TopicExtractionConfig, QueryTopicExtractor

PER_QUERY_TOPICS_CSV = f"data/{STEM}_queries_topics.csv"
TOPIC_NAMES_CSV = f"data/{STEM}_queries_topic_names.csv"
if not (step_done(TOPIC_NAMES_CSV) and step_done(PER_QUERY_TOPICS_CSV)):
    print("Embedding, clustering, and naming topics (this can take a minute or two)...")
    config = TopicExtractionConfig(input_csv=QUERIES_CSV, output_csv=PER_QUERY_TOPICS_CSV, topics_output_csv=TOPIC_NAMES_CSV)
    QueryTopicExtractor(config).run()

df_topic_names = pd.read_csv(TOPIC_NAMES_CSV)
print(f"{len(df_topic_names)} topics found")
df_topic_names.sort_values("num_queries", ascending=False).head(5)

## 4. Reduce topics to a qid/query table

`topic_to_queries.py`'s `TopicQuerySubsetter`.

In [ ]:
from topic_to_queries import TopicSubsetConfig, TopicQuerySubsetter

TOPICS_QID_QUERY_CSV = f"data/{STEM}_queries_topic_names_qid_query.csv"
if not step_done(TOPICS_QID_QUERY_CSV):
    config = TopicSubsetConfig(input_csv=TOPIC_NAMES_CSV, output_csv=TOPICS_QID_QUERY_CSV)
    TopicQuerySubsetter(config).run()

df_all_topics = pd.read_csv(TOPICS_QID_QUERY_CSV, dtype={"qid": str})
print(f"{len(df_all_topics)} topic/query rows")
df_all_topics.head(3)

## 5. Convert chunks to a docno/text index source

`chunks_to_docs.py`'s `ChunkDocsConverter`.

In [ ]:
from chunks_to_docs import ChunkDocsConfig, ChunkDocsConverter

CHUNKS_DOCS_CSV = f"data/{STEM}_chunks_docs.csv"
if not step_done(CHUNKS_DOCS_CSV):
    config = ChunkDocsConfig(input_csv=CHUNKS_CSV, output_csv=CHUNKS_DOCS_CSV)
    ChunkDocsConverter(config).run()

df_docs = pd.read_csv(CHUNKS_DOCS_CSV)
print(f"{len(df_docs)} indexable chunks")
df_docs.head(3)

## 6. (Optional) sample topics for retrieval/evaluation

A fixed, seeded random sample of the full topic set. To evaluate against
*all* topics instead, just set `SAMPLE_TOPICS_CSV = TOPICS_QID_QUERY_CSV`
before running the next cell.

In [ ]:
SAMPLE_SIZE = 30
SAMPLE_SEED = 42
SAMPLE_TOPICS_CSV = f"data/{STEM}_queries_topic_names_qid_query_sample{SAMPLE_SIZE}.csv"
if not step_done(SAMPLE_TOPICS_CSV):
    n = min(SAMPLE_SIZE, len(df_all_topics))
    df_all_topics.sample(n=n, random_state=SAMPLE_SEED).reset_index(drop=True).to_csv(SAMPLE_TOPICS_CSV, index=False)

SAMPLE_STEM = os.path.splitext(SAMPLE_TOPICS_CSV)[0]

df_sample = pd.read_csv(SAMPLE_TOPICS_CSV, dtype={"qid": str})
print(f"{len(df_sample)} sampled topics")
df_sample.head(5)

## 7. Retrieve: BM25, BM25+monoT5, SPLADE (sparse), E5, Qwen3 (dense)

`pt_retrieval.py` / `pt_dense_retrieval.py`. The Danish stemmer is
auto-detected from the docs filename (only matches `fln_praksis_2026*`), so
this needs no dataset-specific branching.

In [ ]:
from pt_retrieval import RetrievalConfig, PyTerrierRetrievalPipeline
from pt_dense_retrieval import DenseRetrievalConfig, DenseRetrievalPipeline

result_paths = {}

for retriever in ["bm25", "bm25_monot5", "splade"]:
    rerank = retriever.endswith("_monot5")
    base_retriever = "bm25" if rerank else retriever
    out_path = f"{SAMPLE_STEM}_{retriever}.csv"
    if not step_done(out_path):
        print(f"Running {retriever} ...")
        config = RetrievalConfig(
            docs_csv=CHUNKS_DOCS_CSV, topics_csv=SAMPLE_TOPICS_CSV,
            retriever=base_retriever, rerank_monot5=rerank, output_csv=out_path,
        )
        PyTerrierRetrievalPipeline(config).run()
    else:
        print(f"Already ran {retriever} -> {out_path} (skipping)")
    result_paths[retriever] = out_path

for retriever in ["e5", "qwen3"]:
    out_path = f"{SAMPLE_STEM}_{retriever}.csv"
    if not step_done(out_path):
        print(f"Running {retriever} ...")
        config = DenseRetrievalConfig(docs_csv=CHUNKS_DOCS_CSV, topics_csv=SAMPLE_TOPICS_CSV, retriever=retriever, output_csv=out_path)
        DenseRetrievalPipeline(config).run()
    else:
        print(f"Already ran {retriever} -> {out_path} (skipping)")
    result_paths[retriever] = out_path

result_paths

## 8. Pool results into one deduplicated candidate set

`pool_results.py`'s `ResultPooler`.

In [ ]:
from pool_results import PoolingConfig, ResultPooler

POOL_CSV = f"{SAMPLE_STEM}_pool.csv"
if not step_done(POOL_CSV):
    config = PoolingConfig(dataset=STEM, results_glob=f"{SAMPLE_STEM}_*.csv", output_csv=POOL_CSV)
    ResultPooler(config).run()
else:
    print(f"Already pooled -> {POOL_CSV} (skipping)")

df_pool = pd.read_csv(POOL_CSV, dtype={"qid": str, "docno": str})
print(f"{len(df_pool)} unique (qid, docno) pairs in the pool")
df_pool.head(3)

## 9. LLM-as-judge relevance labelling

`judge_pool.py`'s `QrelsJudge` (Ollama, `llama3.1:8b` default). This is the
slowest stage on a large pool (potentially hours), so it's skipped whenever
a labels file already exists.

In [ ]:
from judge_pool import JudgeConfig, QrelsJudge

LABELS_CSV = f"{SAMPLE_STEM}_pool_labels.csv"
if not step_done(LABELS_CSV):
    print("Judging the pool with a local LLM (this can take a while)...")
    config = JudgeConfig(input_csv=POOL_CSV, output_csv=LABELS_CSV)
    QrelsJudge(config).run()
else:
    print(f"Already judged -> {LABELS_CSV} (skipping)")

df_labels = pd.read_csv(LABELS_CSV, dtype={"qid": str, "docno": str}).dropna(subset=["label"])
print(f"{len(df_labels)} judged (qid, docno) pairs")
df_labels["label"].astype(int).value_counts().sort_index()

## 10. Evaluate every method: MAP@100, NDCG@10, MRR@10, P@5, R@5

`evaluate_run.py`'s `RunEvaluator`, via `pt.Evaluate`.

In [ ]:
from evaluate_run import EvaluationConfig, RunEvaluator

metrics_rows = []
for retriever, results_path in result_paths.items():
    config = EvaluationConfig(
        dataset=STEM, retriever=retriever, results_csv=results_path, labels_csv=LABELS_CSV,
        output_csv=f"{SAMPLE_STEM}_{retriever}_metrics.csv",
    )
    metrics_rows.append(RunEvaluator(config).run())

metrics_df = pd.concat(metrics_rows, ignore_index=True).sort_values("AP@100", ascending=False).reset_index(drop=True)
metrics_df.to_csv(f"{SAMPLE_STEM}_all_methods_metrics.csv", index=False)
metrics_df